# Ef12: EfficientNetB0 with partial fine-tuning (starts from the pretrained run)

12 views per object, 30 epochs. Part of the paper *3D Object Classification Using Multiple 2D Projection and Convolutional Neural Networks*.

**Run `Ep12` first**: this notebook loads its best checkpoint and fine-tunes the last 30 backbone layers.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running on Google Colab: using the paths defined in the configuration cell.")

In [ ]:
EXPERIMENT_CODE = "Ef12"
EXPERIMENT_NAME = "Ef12 EfficientNetB0 with partial fine-tuning (starts from the pretrained run), 12 views, 30 epochs"
BACKBONE = "efficientnetb0"          # "cnn" | "efficientnetb0" | "mobilenetv2" | "resnet50"
MODE = "finetune"            # "scratch" | "pretrained" | "finetune"
N_VIEWS = 12
EPOCHS = 30
RUN_TAG = ""               # e.g. "_elev10" / "_elev50" for the camera-elevation ablation

DATASET_PATH = f"/content/ModelNet10_{N_VIEWS}views_224x224{RUN_TAG}"
ZIP_PATH = f"/content/drive/MyDrive/ModelNet10_{N_VIEWS}views_224x224{RUN_TAG}.zip"
OUTPUT_DIR = "/content/drive/MyDrive"

IMG_SIZE = 224
CHANNELS = 3
BATCH_SIZE = 64
VAL_RATIO = 0.15
SEED = 42
LEARNING_RATE = 1e-4
FINETUNE_LR = 1e-4          # selected by the hyperparameter search (1e-5 was the original value)
UNFREEZE_LAST_N_LAYERS = 30
PRETRAINED_MODEL_INPUT = f"{OUTPUT_DIR}/Ep12_EfficientNetB0_pretrained_12views_30epochs{RUN_TAG}_BEST.keras"
PRETRAINED_WEIGHTS = "imagenet"   # only used when MODE == "pretrained"

_STEM = f"EfficientNetB0_finetuning_{N_VIEWS}views_{EPOCHS}epochs{RUN_TAG}"
BEST_MODEL_OUTPUT = f"{OUTPUT_DIR}/Ef12_{_STEM}_BEST.keras"
FINAL_MODEL_OUTPUT = f"{OUTPUT_DIR}/Ef12_{_STEM}_FINAL.keras"
HISTORY_OUTPUT = f"{OUTPUT_DIR}/Ef12_history_{_STEM}.csv"
RESULTS_OUTPUT = f"{OUTPUT_DIR}/Ef12_results_{_STEM}.txt"
CONFUSION_MATRIX_OUTPUT = f"{OUTPUT_DIR}/Ef12_confusion_matrix_{_STEM}.csv"

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.backend.clear_session()

print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Per-backbone input preprocessing (images are decoded as float32 in [0, 255]).
# EfficientNet's preprocess_input is the identity: the model rescales internally.
PREPROCESS_FNS = {
    "cnn": lambda x: x / 255.0,
    "efficientnetb0": tf.keras.applications.efficientnet.preprocess_input,
    "mobilenetv2": tf.keras.applications.mobilenet_v2.preprocess_input,
    "resnet50": tf.keras.applications.resnet50.preprocess_input,
}
PREPROCESS_FN = PREPROCESS_FNS[BACKBONE]

BACKBONES = {
    "efficientnetb0": tf.keras.applications.EfficientNetB0,
    "mobilenetv2": tf.keras.applications.MobileNetV2,
    "resnet50": tf.keras.applications.ResNet50,
}

In [ ]:
import zipfile

EXTRACT_TO = os.path.dirname(DATASET_PATH)

if not os.path.exists(DATASET_PATH):
    print("Dataset not found. Extracting from Google Drive...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_TO)
else:
    print("Dataset already exists.")

print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Train exists:", os.path.exists(os.path.join(DATASET_PATH, "train")))
print("Test exists:", os.path.exists(os.path.join(DATASET_PATH, "test")))

if os.path.exists(DATASET_PATH):
    print("Dataset content:", os.listdir(DATASET_PATH))

In [ ]:
def get_classes(dataset_path):
    train_path = os.path.join(dataset_path, "train")
    classes = [
        d for d in os.listdir(train_path)
        if os.path.isdir(os.path.join(train_path, d))
    ]
    classes.sort()
    return classes


def load_objects(dataset_path, split, class_to_idx, n_views):
    split_path = os.path.join(dataset_path, split)
    objects = []

    for class_name, label in class_to_idx.items():
        class_path = os.path.join(split_path, class_name)

        if not os.path.isdir(class_path):
            continue

        object_folders = sorted([
            d for d in os.listdir(class_path)
            if os.path.isdir(os.path.join(class_path, d))
        ])

        for obj_folder in object_folders:
            obj_path = os.path.join(class_path, obj_folder)
            image_paths = sorted([
                os.path.join(obj_path, f)
                for f in os.listdir(obj_path)
                if f.lower().endswith(".png")
            ])

            if len(image_paths) != n_views:
                print(f"Warning: {obj_path} has {len(image_paths)} views and will be ignored.")
                continue

            objects.append({
                "object_path": obj_path,
                "image_paths": image_paths,
                "label": label,
                "class_name": class_name,
            })

    return objects


def split_train_val_objects(train_objects, val_ratio=0.15):
    # Object-level split: all views of an object stay in the same subset.
    train_objects = train_objects.copy()
    random.shuffle(train_objects)

    n_val = int(len(train_objects) * val_ratio)
    val_objects = train_objects[:n_val]
    train_objects = train_objects[n_val:]

    return train_objects, val_objects


def objects_to_image_samples(objects):
    image_paths = []
    labels = []

    for obj in objects:
        for img_path in obj["image_paths"]:
            image_paths.append(img_path)
            labels.append(obj["label"])

    return image_paths, labels


def load_image_tensor(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    return PREPROCESS_FN(image)


def load_image(image_path, label):
    return load_image_tensor(image_path), label


def create_tf_dataset(image_paths, labels, shuffle=True):
    image_paths = tf.constant(image_paths)
    labels = tf.constant(labels, dtype=tf.int32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if shuffle:
        ds = ds.shuffle(
            buffer_size=2048,
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def predict_object(model, object_data):
    # Decision-level fusion: average the class probabilities of all views of the object.
    images = tf.stack([load_image_tensor(p) for p in object_data["image_paths"]], axis=0)
    probs = model.predict(images, verbose=0)
    mean_probs = np.mean(probs, axis=0)
    predicted_label = int(np.argmax(mean_probs))

    return predicted_label, mean_probs


def evaluate_multiview(model, test_objects, class_names, results_output, confusion_matrix_output, experiment_name):
    y_true = []
    y_pred = []

    for idx, obj in enumerate(test_objects, start=1):
        pred_label, _ = predict_object(model, obj)
        y_true.append(obj["label"])
        y_pred.append(pred_label)

        if idx % 50 == 0:
            print(f"Evaluating objects: {idx}/{len(test_objects)}")

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred)

    print(experiment_name)
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(report)

    np.savetxt(confusion_matrix_output, cm, delimiter=",", fmt="%d")

    with open(results_output, "w", encoding="utf-8") as file:
        file.write(f"OBJECT-LEVEL TEST RESULTS - {experiment_name}\n")
        file.write("=============================================\n")
        file.write(f"Accuracy:  {acc:.4f}\n")
        file.write(f"Precision: {precision:.4f}\n")
        file.write(f"Recall:    {recall:.4f}\n")
        file.write(f"F1-score:  {f1:.4f}\n\n")
        file.write("Classes:\n")
        for idx, class_name in enumerate(class_names):
            file.write(f"{idx}: {class_name}\n")
        file.write("\nClassification report:\n")
        file.write(report)
        file.write("\nConfusion matrix:\n")
        file.write(str(cm))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
print("Checking dataset")
print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Train exists:", os.path.exists(os.path.join(DATASET_PATH, "train")))
print("Test exists:", os.path.exists(os.path.join(DATASET_PATH, "test")))

class_names = get_classes(DATASET_PATH)
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

print("Classes:", class_names)
print("Number of classes:", len(class_names))

train_objects_all = load_objects(DATASET_PATH, "train", class_to_idx, N_VIEWS)
test_objects = load_objects(DATASET_PATH, "test", class_to_idx, N_VIEWS)
train_objects, val_objects = split_train_val_objects(train_objects_all, val_ratio=VAL_RATIO)

train_image_paths, train_labels = objects_to_image_samples(train_objects)
val_image_paths, val_labels = objects_to_image_samples(val_objects)

print("Original training objects:", len(train_objects_all))
print("Training objects:", len(train_objects))
print("Validation objects:", len(val_objects))
print("Test objects:", len(test_objects))
print("Training images:", len(train_image_paths))
print("Validation images:", len(val_image_paths))

In [ ]:
def prepare_for_finetuning(model):
    # The backbone is the nested Keras model inside the classifier.
    backbone = next((layer for layer in model.layers if isinstance(layer, tf.keras.Model)), None)

    if backbone is None:
        raise ValueError("Backbone was not found in the loaded model.")

    print("Backbone:", backbone.name)
    print("Total backbone layers:", len(backbone.layers))

    backbone.trainable = True

    for layer in backbone.layers:
        layer.trainable = False

    # Unfreeze the last N layers, keeping BatchNormalization layers frozen.
    for layer in backbone.layers[-UNFREEZE_LAST_N_LAYERS:]:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

    trainable_count = sum(1 for layer in backbone.layers if layer.trainable)
    print("Trainable backbone layers:", trainable_count)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=FINETUNE_LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

if not os.path.exists(PRETRAINED_MODEL_INPUT):
    raise FileNotFoundError(f"Pretrained (frozen-base) model not found: {PRETRAINED_MODEL_INPUT}. Run the corresponding pretrained notebook first.")

model = tf.keras.models.load_model(PRETRAINED_MODEL_INPUT)
model = prepare_for_finetuning(model)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_OUTPUT,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        HISTORY_OUTPUT,
        append=False,
    ),
]

In [ ]:
train_ds = create_tf_dataset(train_image_paths, train_labels, shuffle=True)
val_ds = create_tf_dataset(val_image_paths, val_labels, shuffle=False)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(FINAL_MODEL_OUTPUT)

# The test set is evaluated with the checkpoint that had the best validation accuracy.
best_model = tf.keras.models.load_model(BEST_MODEL_OUTPUT)

evaluate_multiview(
    model=best_model,
    test_objects=test_objects,
    class_names=class_names,
    results_output=RESULTS_OUTPUT,
    confusion_matrix_output=CONFUSION_MATRIX_OUTPUT,
    experiment_name=EXPERIMENT_NAME,
)